In [4]:
#Exercise 0
!pip install pyomo
!pip install cbcbox

import pyomo.environ as pyo

# Create a Pyomo model
model = pyo.ConcreteModel()

# Define the sets
model.plants = pyo.Set(initialize=['Chandigarh', 'Raipur', 'Dharwad', 'Howrah'])
model.markets = pyo.Set(initialize=['Delhi', 'Mumbai', 'Bengaluru', 'Nagpur', 'Hyderabad', 'Chennai', 'Kolkatta', 'Patna', 'Lucknow', 'Ahmedabad'])

# Define the parameters
supply = {
    'Chandigarh': 500,
    'Raipur': 600,
    'Dharwad': 400,
    'Howrah': 700
}

demand = {
    'Delhi': 200,
    'Mumbai': 300,
    'Bengaluru': 250,
    'Nagpur': 150,
    'Hyderabad': 200,
    'Chennai': 200,
    'Kolkatta': 300,
    'Patna': 100,
    'Lucknow': 150,
    'Ahmedabad': 200
}

# Example transportation costs
costs = {
    ('Chandigarh', 'Delhi'): 10, ('Chandigarh', 'Mumbai'): 15, ('Chandigarh', 'Bengaluru'): 25, ('Chandigarh', 'Nagpur'): 20, ('Chandigarh', 'Hyderabad'): 22, ('Chandigarh', 'Chennai'): 28, ('Chandigarh', 'Kolkatta'): 30, ('Chandigarh', 'Patna'): 18, ('Chandigarh', 'Lucknow'): 12, ('Chandigarh', 'Ahmedabad'): 8,
    ('Raipur', 'Delhi'): 18, ('Raipur', 'Mumbai'): 12, ('Raipur', 'Bengaluru'): 20, ('Raipur', 'Nagpur'): 10, ('Raipur', 'Hyderabad'): 15, ('Raipur', 'Chennai'): 22, ('Raipur', 'Kolkatta'): 14, ('Raipur', 'Patna'): 16, ('Raipur', 'Lucknow'): 20, ('Raipur', 'Ahmedabad'): 16,
    ('Dharwad', 'Delhi'): 25, ('Dharwad', 'Mumbai'): 18, ('Dharwad', 'Bengaluru'): 10, ('Dharwad', 'Nagpur'): 15, ('Dharwad', 'Hyderabad'): 12, ('Dharwad', 'Chennai'): 18, ('Dharwad', 'Kolkatta'): 25, ('Dharwad', 'Patna'): 30, ('Dharwad', 'Lucknow'): 22, ('Dharwad', 'Ahmedabad'): 14,
    ('Howrah', 'Delhi'): 30, ('Howrah', 'Mumbai'): 25, ('Howrah', 'Bengaluru'): 28, ('Howrah', 'Nagpur'): 22, ('Howrah', 'Hyderabad'): 20, ('Howrah', 'Chennai'): 15, ('Howrah', 'Kolkatta'): 10, ('Howrah', 'Patna'): 12, ('Howrah', 'Lucknow'): 18, ('Howrah', 'Ahmedabad'): 25
}

model.x = pyo.Var(within=pyo.Integers,bounds=(0,None))
model.z = pyo.Var(model.plants, domain=pyo.Binary)


model.supply = pyo.Param(model.plants, initialize=supply)
model.demand = pyo.Param(model.markets, initialize=demand)
model.costs = pyo.Param(model.plants, model.markets, initialize=costs)

# Define the decision variables
model.x = pyo.Var(model.plants, model.markets, within=pyo.NonNegativeReals)

# Define the objective function
def total_cost(model):
    return sum(model.costs[p, m] * model.x[p, m] for p in model.plants for m in model.markets)
model.objective = pyo.Objective(rule=total_cost, sense=pyo.minimize)

# Define a constraint function
def supply_constraint(model, p):
    return sum(model.x[p, m] for m in model.markets) <= model.supply[p]

# Put the above constraint function in the model
model.supply_constraint = pyo.Constraint(model.plants, rule=supply_constraint)







# define another constraint function
def demand_constraint(model, m):
    return sum(model.x[p, m] for p in model.plants) >= model.demand[m]

# Put these constraints also in the model
model.demand_constraint = pyo.Constraint(model.markets, rule=demand_constraint)

#now solve
solver = pyo.SolverFactory('cbc')
results = solver.solve(model, tee=True)

model.pprint() # Uncomment to print the model

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
'pyomo.core.base.var.ScalarVar'>) on block unknown with a new Component
(type=<class 'pyomo.core.base.var.IndexedVar'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
CBC devel -- COIN-OR Branch and Cut
  args: -printingOptions all -import C:\Users\08042\AppData\Local\Temp\tmp0ynyp2v_.pyomo.lp -stat 1 -solve -solu C:\Users\08042\AppData\Local\Temp\tmp0ynyp2v_.pyomo.soln
  default strategy 1

> Problem loading

  |Rows| = 14   |Cols| = 40   |NZ| = 80
  Variables: 40 continuous

  Coefficient ranges:
    Matrix  in [  1,   1]   ratio =    1
    Cost    in [  8,  30]   ratio = 3.75
    Bounds  in [  -,   -]   ratio =    -
    RHS     in [100, 700]   ratio =    7

+ Problem loaded - 14 rows, 40 cols, 80 NZ
Statistics for presolved model


Problem has 14 rows,